In [1]:
import sys
from pathlib import Path
repo_root = Path.cwd()
sys.path.append(str(repo_root.parent / "src"))


In [2]:
from ingestion.load_mwoz import load_mwoz, quick_stats
import pandas as pd


In [3]:
df_mwoz = load_mwoz("../data/raw/MWOZ.txt")
quick_stats(df_mwoz)
df_mwoz.head()


Rows: 23108
Conversations: 1000
Speakers: {'USER': 12553, 'SYSTEM': 10555}
OVERALL lines: 1000


,conv_id,turn_id,speaker,text,dialog_act,scores_raw,scores,is_overall
0,0,1,USER,I'm looking for a cheap restaurant in the east...,Restaurant-Inform,"3,3,3,3","[3, 3, 3, 3]",False
1,0,2,SYSTEM,the missing sock is a nice restaurant in the e...,Restaurant-Inform,None,None,False
2,0,3,USER,What is the address and phone number?,Restaurant-Request,"3,3,3,3","[3, 3, 3, 3]",False
3,0,4,SYSTEM,The address of The Missing Sock is Finders Cor...,general-reqmore,None,None,False
4,0,5,USER,Does this restaurant have highchairs for babies?,None,"3,3,3,3","[3, 3, 3, 3]",False


In [4]:
# Calculate satisfaction_mean and add label column
def avg_score(score_list):
    if not score_list:
        return None
    return sum(score_list) / len(score_list)

df_mwoz["satisfaction_mean"] = df_mwoz["scores"].apply(avg_score)

# Add label column if dialog_act exists
if "dialog_act" in df_mwoz.columns:
    df_mwoz["label"] = df_mwoz["dialog_act"]


In [5]:
# MWOZ Statistics
import numpy as np

print("=" * 50)
print("MWOZ Dataset Statistics")
print("=" * 50)
print(f"Num rows: {len(df_mwoz)}")
print(f"Num conversations: {df_mwoz['conv_id'].nunique()}")
print(f"Num OVERALL lines: {df_mwoz['is_overall'].sum()}")
print(f"\nSpeakers value counts:")
print(df_mwoz['speaker'].value_counts())
if 'label' in df_mwoz.columns:
    print(f"\nLabel value counts:")
    print(df_mwoz['label'].value_counts())
else:
    print("\nLabel column does not exist")
satisfaction_mean = df_mwoz['satisfaction_mean'].mean()
if pd.notna(satisfaction_mean):
    print(f"\nSatisfaction mean: {satisfaction_mean:.4f}")
else:
    print("\nSatisfaction mean: N/A")
print("\n" + "=" * 50)
print("Preview head():")
print("=" * 50)
df_mwoz.head()


MWOZ Dataset Statistics
Num rows: 23108
Num conversations: 1000
Num OVERALL lines: 1000

Speakers value counts:
speaker
USER      12553
SYSTEM    10555
Name: count, dtype: int64

Label value counts:
label
Hotel-Inform            2425
Restaurant-Inform       2005
Attraction-Inform       1316
Train-Inform            1146
general-thank            992
Hotel-Request            817
general-reqmore          799
Restaurant-Request       725
Train-Request            709
Booking-Book             699
Booking-Request          479
Taxi-Inform              384
Attraction-Request       365
Train-OfferBook          272
Taxi-Request             220
Train-OfferBooked        217
general-bye              208
Restaurant-Recommend     170
Booking-Inform           165
Hotel-NoOffer            145
Restaurant-NoOffer       112
Hotel-Select             111
Restaurant-Select        101
Booking-NoBook            81
Attraction-Recommend      73
general-greet             71
Attraction-NoOffer        70
Hotel-Recomm

,conv_id,turn_id,speaker,text,dialog_act,scores_raw,scores,is_overall,satisfaction_mean,label
0,0,1,USER,I'm looking for a cheap restaurant in the east...,Restaurant-Inform,"3,3,3,3","[3, 3, 3, 3]",False,3.0,Restaurant-Inform
1,0,2,SYSTEM,the missing sock is a nice restaurant in the e...,Restaurant-Inform,None,None,False,NaN,Restaurant-Inform
2,0,3,USER,What is the address and phone number?,Restaurant-Request,"3,3,3,3","[3, 3, 3, 3]",False,3.0,Restaurant-Request
3,0,4,SYSTEM,The address of The Missing Sock is Finders Cor...,general-reqmore,None,None,False,NaN,general-reqmore
4,0,5,USER,Does this restaurant have highchairs for babies?,None,"3,3,3,3","[3, 3, 3, 3]",False,3.0,None


In [6]:
df_mwoz.to_parquet("../data/processed/mwoz_turns.parquet", index=False)
print("Saved mwoz_turns.parquet")


Saved mwoz_turns.parquet
